In [1]:
import os
import shutil
import random

random.seed(42)  # reproducibility — same split every time we run this

clean_dir = r"data\cleaned"
split_dir = r"data\split"
classes = os.listdir(clean_dir)

train_ratio, val_ratio = 0.8, 0.1  # test = remaining 0.1

for cls in classes:
    src_path = os.path.join(clean_dir, cls)
    files = os.listdir(src_path)
    random.shuffle(files)

    n = len(files)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    splits = {
        "train": files[:n_train],
        "val": files[n_train:n_train + n_val],
        "test": files[n_train + n_val:]
    }

    for split_name, split_files in splits.items():
        dst_path = os.path.join(split_dir, split_name, cls)
        os.makedirs(dst_path, exist_ok=True)
        for fname in split_files:
            shutil.copy(os.path.join(src_path, fname), os.path.join(dst_path, fname))

    print(f"{cls}: train={len(splits['train'])}, val={len(splits['val'])}, test={len(splits['test'])}")

acne: train=1619, val=202, test=203
blackheades: train=1058, val=132, test=133
dark spots: train=960, val=120, test=121
pores: train=867, val=108, test=109
wrinkles: train=1287, val=160, test=162


In [2]:
import os
from PIL import Image

split_dir = r"data\split"
resized_dir = r"data\resized"
target_size = (224, 224)

for split_name in ["train", "val", "test"]:
    for cls in os.listdir(os.path.join(split_dir, split_name)):
        src_path = os.path.join(split_dir, split_name, cls)
        dst_path = os.path.join(resized_dir, split_name, cls)
        os.makedirs(dst_path, exist_ok=True)

        for fname in os.listdir(src_path):
            img = Image.open(os.path.join(src_path, fname)).convert("RGB")
            img_resized = img.resize(target_size)
            img_resized.save(os.path.join(dst_path, fname))

    print(f"{split_name} done")

train done
val done
test done


In [3]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.21.0
GPU available: []


In [4]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_dir = r"data\resized\train"
val_dir = r"data\resized\val"
test_dir = r"data\resized\test"

img_size = (224, 224)
batch_size = 32

# Training data: augmentation applied
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    zoom_range=0.1
)

# Val/test: NO augmentation, only rescaling (must reflect real, unmodified data)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir, target_size=img_size, batch_size=batch_size, class_mode="categorical"
)

val_generator = val_test_datagen.flow_from_directory(
    val_dir, target_size=img_size, batch_size=batch_size, class_mode="categorical"
)

test_generator = val_test_datagen.flow_from_directory(
    test_dir, target_size=img_size, batch_size=batch_size, class_mode="categorical"
)

Found 5791 images belonging to 5 classes.
Found 722 images belonging to 5 classes.
Found 728 images belonging to 5 classes.
